# Historical Weather Data Analysis

**Dataset:** Climate of England – Professional Reforms

**Before running:** Download the CSV from the Kaggle Input section, rename it to `weather_data.csv`, and place it beside this notebook.

## Research questions
1. What is the distribution and spread of temperature?
2. How is humidity distributed?
3. What is the distribution and spread of wind speed?
4. Is temperature associated with humidity?
5. Is wind speed associated with humidity?

**Dataset limitation:** The supplied reference notebook describes UK weather observations but does not show a country/place column. Therefore, a country comparison cannot be performed unless the downloaded CSV contains a suitable location field.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
DATA_PATH = Path("weather_data.csv")
df = pd.read_csv(DATA_PATH)
df.head()


The first five rows provide a quick preview of the observations and help verify that the CSV was loaded correctly.

In [ ]:
df.info()


The information summary shows the number of records, column names, data types and non-null counts. Use it to identify columns that need conversion or cleaning.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.describe(include="all").T)


The shape reports the dataset size. The descriptive summary gives numerical distributions and category frequencies.

In [ ]:
date_col = "Formatted Date" if "Formatted Date" in df.columns else None
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True).dt.tz_convert(None)
    print("First date:", df[date_col].min())
    print("Last date:", df[date_col].max())
else:
    print("No recognized date column found.")

place_candidates = [c for c in df.columns if any(k in c.lower() for k in ["country", "city", "location", "place"])]
print("Possible place columns:", place_candidates if place_candidates else "None found")


The date column is converted to datetime so chronological operations work correctly. The location check identifies whether a country/place comparison is supported.

In [ ]:
# Keep the columns needed for the research questions and core weather analysis
preferred = ["Formatted Date", "Temperature (C)", "Humidity", "Wind Speed (km/h)",
             "Pressure (millibars)", "Visibility (km)", "Precip Type", "Summary", "Daily Summary"]
selected = [c for c in preferred if c in df.columns]
weather = df[selected].copy()
print("Selected columns:", selected)
weather.head()


Only relevant date and weather fields are retained for the main analysis. Columns are selected only when they exist in the downloaded file.

In [ ]:
# Missing values: counts and visualization
missing = weather.isna().sum().sort_values(ascending=False)
display(missing.to_frame("Missing values"))
fig, ax = plt.subplots(figsize=(10, 4))
missing.plot(kind="bar", ax=ax)
ax.set(title="Missing values by column", xlabel="Column", ylabel="Missing count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


This chart compares missing-value counts across selected columns. Missing values should be handled according to the meaning of each field, not filled blindly.

In [ ]:
# Missing-value treatment decisions
numeric_weather = [c for c in ["Temperature (C)", "Humidity", "Wind Speed (km/h)",
                               "Pressure (millibars)", "Visibility (km)"] if c in weather.columns]
for col in numeric_weather:
    weather[col] = weather[col].fillna(weather[col].median())
# Keep categorical/date missingness explicit rather than inventing values
for col in ["Precip Type", "Summary", "Daily Summary"]:
    if col in weather.columns:
        weather[col] = weather[col].fillna("Unknown")
print("Remaining missing values:")
display(weather.isna().sum().to_frame("Missing after treatment"))


Numeric weather measurements are imputed with their median because it is less sensitive to extreme values than the mean. Missing category values are labelled `Unknown`; invalid dates are not fabricated.

In [ ]:
# Duplicate checks
print("Exact duplicate rows:", weather.duplicated().sum())
if date_col and date_col in weather.columns:
    print("Repeated timestamps:", weather.duplicated(subset=[date_col]).sum())


Exact duplicate rows are counted. This dataset may contain multiple observations per day or timestamp, so repeated dates alone are not automatically errors; a place-date uniqueness test is only valid if a place column exists.

In [ ]:
# Remove exact duplicate rows only
before = len(weather)
weather = weather.drop_duplicates().copy()
print("Rows removed:", before - len(weather))
print("Rows remaining:", len(weather))


Only exact duplicate rows are removed. Repeated timestamps are retained unless the dataset documentation confirms that they are invalid.

In [ ]:
# Box plots and histograms for temperature, wind speed and rainfall/precipitation
plot_cols = [c for c in ["Temperature (C)", "Wind Speed (km/h)"] if c in weather.columns]
for col in plot_cols:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.boxplot(x=weather[col], ax=axes[0])
    axes[0].set(title=f"{col}: box plot", xlabel=col)
    sns.histplot(weather[col], bins=30, kde=True, ax=axes[1])
    axes[1].set(title=f"{col}: histogram", xlabel=col, ylabel="Frequency")
    plt.tight_layout()
    plt.show()

print("No direct rainfall amount column is assumed. Precip Type is categorical, not a rainfall measurement.")


Box plots highlight potential extreme values, while histograms show the overall distribution. The provided schema contains precipitation type rather than a numeric rainfall amount, so rainfall amount plots are not applicable.

In [ ]:
# IQR outlier check for at least two numeric columns
def iqr_outlier_summary(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < low) | (series > high)]
    return {"Q1": q1, "Q3": q3, "IQR": iqr, "Lower bound": low,
            "Upper bound": high, "Outlier count": len(outliers)}

for col in [c for c in ["Temperature (C)", "Wind Speed (km/h)", "Humidity"] if c in weather.columns][:3]:
    print(col, iqr_outlier_summary(weather[col].dropna()))


The IQR rule flags observations below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. These are potential outliers, not automatic errors; weather extremes can be genuine events and should be checked before removal.

In [ ]:
# Univariate analysis: distribution, typical value and spread
for col in [c for c in ["Temperature (C)", "Humidity", "Wind Speed (km/h)",
                         "Pressure (millibars)", "Visibility (km)"] if c in weather.columns]:
    s = weather[col].dropna()
    print(f"{col}: median={s.median():.3f}, mean={s.mean():.3f}, standard deviation={s.std():.3f}, range={s.min():.3f} to {s.max():.3f}")
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(s, bins=30, kde=True, ax=ax)
    ax.set(title=f"Distribution of {col}", xlabel=col, ylabel="Frequency")
    plt.tight_layout()
    plt.show()


Each histogram shows one weather variable at a time. The median is used as a robust typical value, while standard deviation and range describe spread.

In [ ]:
# Bivariate analysis: temperature vs humidity
if {"Temperature (C)", "Humidity"}.issubset(weather.columns):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=weather, x="Temperature (C)", y="Humidity", alpha=0.25, ax=ax)
    ax.set(title="Temperature vs humidity", xlabel="Temperature (°C)", ylabel="Humidity")
    plt.tight_layout()
    plt.show()
    print("Pearson correlation:", weather[["Temperature (C)", "Humidity"]].corr().iloc[0, 1])


The scatter plot shows how humidity varies with temperature. Pearson correlation summarizes the direction and strength of a linear association; it does not prove causation.

In [ ]:
# Bivariate analysis: wind speed vs humidity
if {"Wind Speed (km/h)", "Humidity"}.issubset(weather.columns):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=weather, x="Wind Speed (km/h)", y="Humidity", alpha=0.25, ax=ax)
    ax.set(title="Wind speed vs humidity", xlabel="Wind speed (km/h)", ylabel="Humidity")
    plt.tight_layout()
    plt.show()


This scatter plot checks whether higher wind speeds tend to occur with different humidity levels. The pattern should be interpreted from the actual observations.

In [ ]:
# Correlation heatmap
corr_cols = weather.select_dtypes(include=np.number).columns
if len(corr_cols) >= 2:
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(weather[corr_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
    ax.set_title("Correlation heatmap of weather variables")
    plt.tight_layout()
    plt.show()


The heatmap summarizes pairwise linear correlations among numerical weather variables. Values close to +1 indicate positive association, values close to −1 negative association, and values near 0 weak linear association.

In [ ]:
# Line chart over time: monthly average temperature
if date_col and "Temperature (C)" in weather.columns:
    monthly = (weather.dropna(subset=[date_col]).set_index(date_col)["Temperature (C)"]
               .resample("MS").mean())
    fig, ax = plt.subplots(figsize=(12, 4))
    monthly.plot(ax=ax)
    ax.set(title="Monthly average temperature over time", xlabel="Date", ylabel="Temperature (°C)")
    plt.tight_layout()
    plt.show()


Monthly averages make long hourly time series easier to read and reveal broad seasonal or long-term patterns.

In [ ]:
# Compare average weather category values where categories exist
if "Precip Type" in weather.columns and "Temperature (C)" in weather.columns:
    comparison = weather.groupby("Precip Type", dropna=False)["Temperature (C)"].mean().sort_values()
    display(comparison.to_frame("Average temperature (°C)"))
    ax = comparison.plot(kind="bar", figsize=(8, 4), title="Average temperature by precipitation type")
    ax.set_xlabel("Precipitation type")
    ax.set_ylabel("Average temperature (°C)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
elif "Summary" in weather.columns and "Temperature (C)" in weather.columns:
    display(weather.groupby("Summary")["Temperature (C)"].mean().sort_values().to_frame("Average temperature (°C)"))


The category comparison replaces a country comparison because the supplied dataset schema does not identify multiple countries. It compares average temperature across precipitation types when those fields are available.

In [ ]:
# Answer the five research questions with actual values
if "Temperature (C)" in weather.columns:
    s = weather["Temperature (C)"]
    print("Q1. Temperature distribution:")
    print(f"Median={s.median():.2f} °C; standard deviation={s.std():.2f} °C; range={s.min():.2f} to {s.max():.2f} °C.")
if "Humidity" in weather.columns:
    s = weather["Humidity"]
    print("\nQ2. Humidity distribution:")
    print(f"Median={s.median():.3f}; standard deviation={s.std():.3f}; range={s.min():.3f} to {s.max():.3f}.")
if "Wind Speed (km/h)" in weather.columns:
    s = weather["Wind Speed (km/h)"]
    print("\nQ3. Wind speed distribution:")
    print(f"Median={s.median():.2f} km/h; standard deviation={s.std():.2f} km/h; range={s.min():.2f} to {s.max():.2f} km/h.")
if {"Temperature (C)", "Humidity"}.issubset(weather.columns):
    print("\nQ4. Temperature vs humidity correlation:",
          round(weather[["Temperature (C)", "Humidity"]].corr().iloc[0, 1], 3))
if {"Wind Speed (km/h)", "Humidity"}.issubset(weather.columns):
    print("\nQ5. Wind speed vs humidity correlation:",
          round(weather[["Wind Speed (km/h)", "Humidity"]].corr().iloc[0, 1], 3))


The printed statistics provide concise answers to the five research questions. Use the graphs above alongside these values when writing your interpretation.

In [ ]:
# Conclusion helper: inspect these values and write a 5–6 line conclusion in your own words
summary_cols = [c for c in ["Temperature (C)", "Humidity", "Wind Speed (km/h)"] if c in weather.columns]
display(weather[summary_cols].describe().T)


## Conclusion

After running the notebook, write 5–6 lines here using your actual results. Mention the typical temperature, humidity and wind speed; the main distribution/outlier observations; the temperature–humidity relationship; the wind–humidity relationship; and one limitation (no country/place field and no numeric rainfall amount if confirmed by the CSV).